In [7]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when, trim, regexp_replace

# ==========================================
# 1. SETUP (Load Delta)
# ==========================================
# Stop any old session first
try:
    SparkSession.builder.getOrCreate().stop()
except:
    pass

spark = (SparkSession.builder
    .appName("GDP Fixed Pipeline")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") 
    .config("spark.sql.warehouse.dir", "hdfs://hdfs-nn:9000/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

print("✅ Spark Session Started.")

# ==========================================
# 2. READ & CLEAN COLUMN NAMES
# ==========================================
bronze_path = "hdfs://hdfs-nn:9000/demo/bronze/gdp_ppp_per_capita.csv"

# Read
df_raw = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Clean Column Names (e.g., "Country Name" -> "country_name")
for col_name in df_raw.columns:
    clean_name = col_name.strip().replace(" ", "_").lower()
    df_raw = df_raw.withColumnRenamed(col_name, clean_name)

print("Columns found:", df_raw.columns[:5], "...") 

# ==========================================
# 3. TRANSFORM: UNPIVOT (Wide -> Long)
# ==========================================
# This is the MAGIC STEP you were missing.
# We take all columns that are NOT 'country_name' and stack them into rows.

# Identify which columns are years (everything except country columns)
non_year_cols = [c for c in df_raw.columns if "country" in c]
year_cols = [c for c in df_raw.columns if c not in non_year_cols]

print(f"Transforming {len(year_cols)} year columns into rows...")

# Create the SQL expression to unpivot
# This turns columns `1990`, `1991` into rows under column `year`
stack_expr = f"""stack({len(year_cols)}, {', '.join([f"'{c}', `{c}`" for c in year_cols])}) as (year, gdp_per_capita)"""

# Apply the transformation
df_unpivoted = df_raw.select(
    *non_year_cols, 
    expr(stack_expr)
)

# Final Data Type cleanup
df_clean = df_unpivoted \
    .withColumn("year", col("year").cast("int")) \
    .withColumn("gdp_per_capita", col("gdp_per_capita").cast("double")) \
    .filter(col("gdp_per_capita").isNotNull()) # Remove empty values

print("✅ Transformation Complete. Schema is now:")
df_clean.printSchema()
df_clean.show(5)

# ==========================================
# 4. WRITE TO SILVER (FIXED)
# ==========================================
silver_path = "hdfs://hdfs-nn:9000/warehouse/silver.db/gdp"

print(f"Saving to {silver_path} (Partitioned by Year)...")

(df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # <--- THIS IS THE MAGIC FIX
    .partitionBy("year")
    .save(silver_path)
)
print("✅ Saved to HDFS with new Schema.")
# ==========================================
# 5. REGISTER IN HIVE
# ==========================================
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db'")

# Drop the old reference entirely
spark.sql("DROP TABLE IF EXISTS silver.gdp")

# Create the new reference
spark.sql(f"""
    CREATE TABLE silver.gdp
    USING DELTA
    LOCATION '{silver_path}'
""")

# Repair is usually automatic for Delta, but good practice
try:
    spark.sql("MSCK REPAIR TABLE silver.gdp")
except:
    pass

print("✅ SUCCESS! Table 'silver.gdp' is fully fixed and registered.")

✅ Spark Session Started.
Columns found: ['country_name', '1990', '1991', '1992', '1993'] ...
Transforming 31 year columns into rows...
✅ Transformation Complete. Schema is now:
root
 |-- country_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gdp_per_capita: double (nullable = true)

+------------+----+--------------+
|country_name|year|gdp_per_capita|
+------------+----+--------------+
|       Aruba|1990|     23293.037|
|       Aruba|1991|     25004.172|
|       Aruba|1992|     25643.084|
|       Aruba|1993|      26515.89|
|       Aruba|1994|     27695.545|
+------------+----+--------------+
only showing top 5 rows

Saving to hdfs://hdfs-nn:9000/warehouse/silver.db/gdp (Partitioned by Year)...
✅ Saved to HDFS with new Schema.
✅ SUCCESS! Table 'silver.gdp' is fully fixed and registered.
